In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# As we are having all the data in form of 
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
# from xgboost import XGBRegressor
import warnings

In [ ]:
df=pd.read_csv('data/StudentsPerformance.csv')

In [ ]:
df.head()

In [ ]:
# Preparing X, y variables(Independent and Dependent features)
# Here we consider math score as the o/p feature
X = df.drop(columns='math_score', axis=1)


In [ ]:
y = df['math_score']

In [ ]:
y

In [ ]:
# Create column transformer with 3 types transformers
num_features=X.select_dtypes(exclude='object').columns
cat_features=X.select_dtypes(include='object').columns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

# column transformer combines both one hot encoding and standard scaler and gives the final result.

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
        # as we have only two categories for 'test preparation course' we can use the one hot encoding, if we had higher number of categories then we can use target guided ordinal encoding.
    ]
)

In [ ]:
X = preprocessor.fit_transform(X)


In [ ]:
from sklearn.model_selection import train_test_split
# seperate dataset into train and test
X_train, X_test , y_train, y_test= train_test_split(X, y, test_size =0.2, random_state = 42)
X_train.shape , X_test.shape

In [ ]:
# creating an evaluate function to give all metrics after model training
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(), 
    "Ridge": Ridge(), 
    "K-Neighbors Regressor": KNeighborsRegressor(), 
    "Decision Tree":DecisionTreeRegressor(), 
    "Random Forest Regressor": RandomForestRegressor(), 
    # "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # evaluate train and test dataset
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2= evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print("Model performance for training set")
    print("- Root mean squared error: {:.4f}".format(model_train_rmse))
    print("- Mean absolute error: {:.4f}".format(model_train_mae))
    print("- R2 score: {:.4f}".format(model_train_r2))

    print('-'*35)

    print("Model performance for test set")
    print("- Root mean squared error: {:.4f}".format(model_test_rmse))
    print("- Mean absolute error: {:.4f}".format(model_test_mae))
    print("- R2 score: {:.4f}".format(model_test_r2))

    r2_list.append(model_test_r2)

    print('='*35)
    print('\n')
    



In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)),columns=['Model Name', 'r2_score']).sort_values(by=["r2_score"],ascending=False)

In [ ]:
# Linear Regression is the best performing one
lin_model= LinearRegression(fit_intercept=True)
lin_model = lin_model.fit(X_train,y_train)
y_pred = lin_model.predict(X_test)
score= r2_score(y_test, y_pred)*100
print(" Accuracy of the model is %.2f" %score)


In [ ]:
# Polt y_pred and y_test
plt.scatter(y_test, y_pred)
plt.xlabel('Actual')
plt.ylabel('Predicted')

In [ ]:
sns.regplot(x=y_test,y=y_pred,ci=None, color='red')

In [ ]:
# Difference between actual and predicted values
pred_df = pd.DataFrame({'Actual value':y_test, 'Predicted value':y_pred, 'Difference':y_test-y_pred})
pred_df